In [1]:
import pandas as pd
import numpy as np
import os
import joblib
import warnings

# Suppress sklearn's minor feature name warnings for clean output
warnings.filterwarnings('ignore')

print("--- SENTRi-X: BoT-IoT Cross-Validation Pipeline ---")

# Define our directory paths
processed_dir = "../data/processed/"
models_dir = "../models/"
raw_data_path = "../data/raw/bot_iot/bot_iot_mapped.csv"

print("\n--- Step 1: Loading and Sampling Data ---")
print("Loading the unified BoT-IoT dataset...")
df = pd.read_csv(raw_data_path)
print(f"Original Dataset Size: {len(df):,} packets.")

# We sample down to 200,000 to keep the evaluation lightning-fast and save RAM
print("Sampling down to 200,000 network flows for evaluation...")
df = df.sample(n=200000, random_state=42)

# Clean up any lingering dirt
df = df.drop_duplicates()
df = df.dropna()

# Separate the payload (X) from the answer key (y)
X = df.drop('label', axis=1)
y = df['label']
print(f"Data successfully split. X shape: {X.shape}, y shape: {y.shape}")

--- SENTRi-X: BoT-IoT Cross-Validation Pipeline ---

--- Step 1: Loading and Sampling Data ---
Loading the unified BoT-IoT dataset...
Original Dataset Size: 3,668,522 packets.
Sampling down to 200,000 network flows for evaluation...
Data successfully split. X shape: (144728, 12), y shape: (144728,)


In [2]:
from sklearn.preprocessing import StandardScaler
import joblib
import os
import pandas as pd

print("--- Step 2 & 3: Encoding & Domain-Specific Scaling ---")

# 1. One-Hot Encoding
print("Applying One-Hot Encoding...")
X_encoded = pd.get_dummies(X, columns=['proto', 'conn_state'])

# 2. Align to ToN-IoT schema
rf_model = joblib.load(os.path.join(models_dir, "rf_model_ton_iot.joblib"))
expected_columns = rf_model.feature_names_in_
X_aligned = X_encoded.reindex(columns=expected_columns, fill_value=0)

# 3. Domain Adaptation: Local Scaling
# We use a fresh scaler so BoT-IoT data is centered around its own normal baseline
print("Applying Local Standard Scaling for Domain Adaptation...")
local_scaler = StandardScaler()
X_scaled = local_scaler.fit_transform(X_aligned)

print(f"Final Matrix Shape ready for Transfer Learning: {X_scaled.shape}")

--- Step 2 & 3: Encoding & Domain-Specific Scaling ---
Applying One-Hot Encoding...
Applying Local Standard Scaling for Domain Adaptation...
Final Matrix Shape ready for Transfer Learning: (144728, 28)


In [3]:
print("--- Step 4: Save the Evaluation Matrices ---")

print("Saving the finalized BoT-IoT Evaluation Tensors...")
joblib.dump(X_scaled, os.path.join(processed_dir, "bot_iot_X_test.pkl"))
joblib.dump(y.values, os.path.join(processed_dir, "bot_iot_y_test.pkl"))

# Save a clean DataFrame version just in case we want to run SHAP/LIME on BoT-IoT later
joblib.dump(X_aligned, os.path.join(processed_dir, "bot_iot_X_test_df.pkl"))

print(f"\nSUCCESS! BoT-IoT is fully processed and matrix-aligned.")
print(f"Final Matrix Shape ready for SENTRi-X: {X_scaled.shape}")

--- Step 4: Save the Evaluation Matrices ---
Saving the finalized BoT-IoT Evaluation Tensors...

SUCCESS! BoT-IoT is fully processed and matrix-aligned.
Final Matrix Shape ready for SENTRi-X: (144728, 28)
